# 📈 Pandas en Google Colab · Parte 3 — Uso estadístico con Python

**Unidad 6: regresión lineal simple y múltiple.** Basado en los ejemplos `12` a `15` del repositorio **CursoPython**.

**Pregunta que guía la unidad:** *¿qué explica la nota final de un estudiante y cuánto?*

| Sección | Tema | Ejemplo del repo |
|---|---|---|
| 1 | Correlación y diagramas de dispersión | `12_correlacion.py` |
| 2 | Regresión lineal simple | `13_regresion_simple.py` |
| 3 | Regresión lineal múltiple | `14_regresion_multiple.py` |
| 4 | Predecir y evaluar con datos nuevos | `15_prediccion_evaluacion.py` |
| — | Ejercicios guiados + soluciones | `ejercicios_parte3.py` |

Colab ya trae instalados `statsmodels`, `scikit-learn` y `matplotlib`.

## 0. Cargar los datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

URL = ("https://raw.githubusercontent.com/hpizarrog-industry40/CursoPython/"
       "main/Aprendiendo_Pandas/datos/rendimiento.csv")
df = pd.read_csv(URL)
print(df.shape)
df.head()

| Columna | Descripción |
|---|---|
| `horas_estudio` | Horas de estudio semanales |
| `asistencia` | Porcentaje de asistencia |
| `horas_sueno` | Horas de sueño por noche (**4 faltantes**) |
| `trabaja` | "Si" / "No" |
| `curso` | Matematicas, Fisica o Quimica |
| `nota_final` | Nota final (1–10) — **variable a explicar** |

In [ ]:
df.describe().round(2)

---
## 1. Correlación
`r` va de −1 a 1. Mide la fuerza de una relación **lineal**. Correlación no implica causalidad.

In [ ]:
numericas = ["horas_estudio", "asistencia", "horas_sueno", "nota_final"]
df[numericas].corr().round(2)

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for eje, col in zip(ejes, ["horas_estudio", "asistencia", "horas_sueno"]):
    eje.scatter(df[col], df["nota_final"], alpha=0.6)
    eje.set_xlabel(col)
    eje.set_title(f"r = {df[col].corr(df['nota_final']):.2f}")
ejes[0].set_ylabel("nota_final")
plt.tight_layout(); plt.show()

**Lectura:** horas de estudio (r = 0,80) y asistencia (r = 0,61) se relacionan con la nota; las horas de sueño, casi nada (r = −0,07).
Ojo: horas de estudio y asistencia también se relacionan **entre sí** (r = 0,62). Eso importará en la sección 3.

---
## 2. Regresión lineal simple
Buscamos la recta `nota_final = b0 + b1 · horas_estudio` que minimiza los errores al cuadrado.

In [ ]:
modelo = smf.ols("nota_final ~ horas_estudio", data=df).fit()
b0, b1 = modelo.params["Intercept"], modelo.params["horas_estudio"]
print(f"nota_final = {b0:.2f} + {b1:.3f} * horas_estudio")
print(f"R2 = {modelo.rsquared:.3f}   p-valor pendiente = {modelo.pvalues['horas_estudio']:.1e}")

**Interpretación**
- **Pendiente (0,40):** cada hora extra de estudio semanal se asocia con +0,40 puntos de nota, en promedio.
- **Intercepto (2,72):** nota esperada con 0 horas de estudio (extrapolación: casi nadie estudia 0 horas).
- **R² (0,64):** las horas de estudio explican el 64 % de la variación de la nota.
- **p-valor < 0,05:** la relación es estadísticamente significativa.

In [ ]:
modelo.summary()

In [ ]:
# Comprobación a mano: b1 = cov(x, y) / var(x)
x, y = df["horas_estudio"], df["nota_final"]
print("b1 manual:", round(x.cov(y) / x.var(), 3))

In [ ]:
nuevos = pd.DataFrame({"horas_estudio": [3, 7, 12]})
nuevos["nota_predicha"] = modelo.predict(nuevos).round(2)
nuevos

### Residuos
Residuo = real − predicho. Si el modelo es adecuado, los residuos forman una nube sin patrón alrededor de 0.

In [ ]:
fig, (e1, e2) = plt.subplots(1, 2, figsize=(12, 4))
e1.scatter(x, y, alpha=0.6)
orden = x.sort_values()
e1.plot(orden, b0 + b1 * orden, color="red")
e1.set_xlabel("horas_estudio"); e1.set_ylabel("nota_final"); e1.set_title("Recta ajustada")
e2.scatter(modelo.fittedvalues, modelo.resid, alpha=0.6)
e2.axhline(0, color="red")
e2.set_xlabel("nota predicha"); e2.set_ylabel("residuo"); e2.set_title("Residuos vs. predichos")
plt.tight_layout(); plt.show()

---
## 3. Regresión lineal múltiple
### 3.1 Preparar los datos (Unidad 3)
`horas_sueno` tiene 4 faltantes. **Decisión documentada:** excluir esas 4 filas (menos del 3 %).

In [ ]:
datos = df.dropna(subset=["horas_sueno"])
print(len(df), "->", len(datos), "filas")

### 3.2 De menos a más variables

In [ ]:
formulas = {
    "M1 simple":       "nota_final ~ horas_estudio",
    "M2 + asistencia": "nota_final ~ horas_estudio + asistencia",
    "M3 completo":     "nota_final ~ horas_estudio + asistencia + horas_sueno + C(trabaja) + C(curso)",
}
modelos = {n: smf.ols(f, data=datos).fit() for n, f in formulas.items()}
pd.DataFrame({
    "R2": [m.rsquared for m in modelos.values()],
    "R2_ajustado": [m.rsquared_adj for m in modelos.values()],
    "coef_horas_estudio": [m.params["horas_estudio"] for m in modelos.values()],
}, index=modelos.keys()).round(3)

**Dos lecciones:**
1. El **R² ajustado** penaliza variables que no aportan: úsalo para comparar modelos.
2. El efecto de `horas_estudio` **baja** de 0,40 a 0,31: en el modelo simple también "absorbía" el efecto de asistir y de no trabajar.

### 3.3 Leer los coeficientes

In [ ]:
m3 = modelos["M3 completo"]
pd.DataFrame({"coeficiente": m3.params, "p_valor": m3.pvalues,
              "significativa": m3.pvalues < 0.05}).round(4)

**Interpretación** (*manteniendo constante el resto*):
- `horas_estudio`: +1 h → +0,31 puntos · `asistencia`: +10 % → +0,39 puntos
- `C(trabaja)[T.Si]`: quienes trabajan, −0,86 puntos
- `C(curso)[T.Quimica]`: −0,44 respecto de **Física** (categoría de referencia, primera en orden alfabético)
- `horas_sueno`: p = 0,65 → **no aporta** · `C(curso)[T.Matematicas]`: p = 0,055, al límite

### 3.4 Multicolinealidad (VIF)
Regla práctica: VIF > 5 merece atención; VIF > 10 es grave.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = m3.model.exog
pd.Series([variance_inflation_factor(X, i) for i in range(X.shape[1])],
          index=m3.model.exog_names).drop("Intercept").round(2)

### 3.5 Modelo final y predicción

In [ ]:
final = smf.ols("nota_final ~ horas_estudio + asistencia + C(trabaja) + C(curso)", data=datos).fit()
print(f"R2 ajustado sin horas_sueno: {final.rsquared_adj:.3f}")
estudiante = pd.DataFrame({"horas_estudio": [8], "asistencia": [90],
                           "trabaja": ["No"], "curso": ["Matematicas"]})
print("Nota predicha:", round(final.predict(estudiante).iloc[0], 2))

---
## 4. Predecir y evaluar con datos nuevos (`scikit-learn`)
Separamos 80 % para entrenar y 20 % para probar con estudiantes que el modelo **no vio**.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X = pd.get_dummies(df[["horas_estudio", "asistencia", "trabaja", "curso"]],
                   columns=["trabaja", "curso"], drop_first=True, dtype=int)
y = df["nota_final"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg = LinearRegression().fit(X_train, y_train)
pred = reg.predict(X_test)
print(f"MAE  = {mean_absolute_error(y_test, pred):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.2f}")
print(f"R2   = {r2_score(y_test, pred):.2f}")
print(f"MAE línea base (predecir el promedio) = "
      f"{mean_absolute_error(y_test, np.full(len(y_test), y_train.mean())):.2f}")

In [ ]:
plt.scatter(y_test, pred)
plt.plot([1, 10], [1, 10], color="red")
plt.xlabel("nota real"); plt.ylabel("nota predicha"); plt.title("Predicho vs. real (datos de prueba)")
plt.show()

---
## ✍️ Ejercicios guiados (`ejercicios_parte3.py`)
Cada función trae una **guía** paso a paso. Reemplaza `pass` y ejecuta la celda de comprobación.

In [ ]:
def ejercicio_1_correlacion(df):
    """Correlación entre 'asistencia' y 'nota_final', redondeada a 2 decimales.
    Guía: df["asistencia"].corr(df["nota_final"]) y luego round(..., 2)."""
    pass  # TODO

def ejercicio_2_regresion_simple(df):
    """Ajusta nota_final ~ asistencia y devuelve {"intercepto", "pendiente", "r2"}.
    Guía: smf.ols(...).fit(); modelo.params[...]; modelo.rsquared."""
    pass  # TODO

def ejercicio_3_predecir(df, asistencia):
    """Nota predicha (float) para la asistencia indicada, con el modelo del ejercicio 2.
    Guía: modelo.predict(pd.DataFrame({"asistencia": [asistencia]})).iloc[0]."""
    pass  # TODO

def ejercicio_4_regresion_multiple(df):
    """Modelo nota_final ~ horas_estudio + asistencia + horas_sueno + C(trabaja),
    solo con filas sin faltantes en horas_sueno. Devuelve el modelo ajustado."""
    pass  # TODO

def ejercicio_5_variables_no_significativas(modelo):
    """Lista de variables con p-valor >= 0.05, sin 'Intercept'.
    Guía: modelo.pvalues, filtrar, quitar 'Intercept', .index.tolist()."""
    pass  # TODO

In [ ]:
# Comprobación automática (no modificar)
def test_1(): assert ejercicio_1_correlacion(df) == 0.61
def test_2():
    r = ejercicio_2_regresion_simple(df)
    assert abs(r["pendiente"] - 0.0958) < 0.001 and abs(r["intercepto"] + 2.229) < 0.01
    assert abs(r["r2"] - 0.371) < 0.001
def test_3(): assert abs(ejercicio_3_predecir(df, 90) - 6.39) < 0.01
def test_4():
    m = ejercicio_4_regresion_multiple(df)
    assert int(m.nobs) == 146 and abs(m.params["horas_estudio"] - 0.303) < 0.01
def test_5():
    m = smf.ols("nota_final ~ horas_estudio + asistencia + horas_sueno + C(trabaja)",
                data=df.dropna(subset=["horas_sueno"])).fit()
    assert ejercicio_5_variables_no_significativas(m) == ["horas_sueno"]

def comprobar():
    for n, t in enumerate([test_1, test_2, test_3, test_4, test_5], start=1):
        try:
            t(); print(f"Ejercicio {n}: ✅ OK")
        except Exception as e:
            print(f"Ejercicio {n}: ⏳ pendiente o incorrecto ({type(e).__name__})")

comprobar()

### 📝 Interpretación (completa esta celda)
- Por cada punto adicional de asistencia, la nota final cambia en … puntos.
- La asistencia sola explica el … % de la variación de la nota.
- En el modelo múltiple, la variable que no aporta es … porque su p-valor es …

---
## ✅ Soluciones (revisión en clase)
> 💡 Colapsa este título para ocultar las soluciones hasta la revisión.

In [ ]:
def ejercicio_1_correlacion(df):
    return round(df["asistencia"].corr(df["nota_final"]), 2)

def ejercicio_2_regresion_simple(df):
    m = smf.ols("nota_final ~ asistencia", data=df).fit()
    return {"intercepto": m.params["Intercept"], "pendiente": m.params["asistencia"], "r2": m.rsquared}

def ejercicio_3_predecir(df, asistencia):
    m = smf.ols("nota_final ~ asistencia", data=df).fit()
    return float(m.predict(pd.DataFrame({"asistencia": [asistencia]})).iloc[0])

def ejercicio_4_regresion_multiple(df):
    datos = df.dropna(subset=["horas_sueno"])
    return smf.ols("nota_final ~ horas_estudio + asistencia + horas_sueno + C(trabaja)", data=datos).fit()

def ejercicio_5_variables_no_significativas(modelo):
    p = modelo.pvalues.drop("Intercept", errors="ignore")
    return p[p >= 0.05].index.tolist()

comprobar()

**Resultados esperados:** (1) r = 0,61 · (2) nota = −2,23 + 0,096 · asistencia, R² = 0,37 ·
(3) 90 % de asistencia → 6,39 · (4) 146 filas, horas_estudio = 0,30 · (5) `['horas_sueno']`.

**Interpretación:** cada punto de asistencia suma ~0,10 puntos (10 % → ~1 punto). La asistencia sola explica el 37 %
de la variación; el modelo múltiple llega a ~72 % (R² ajustado). Las horas de sueño no aportan (p = 0,93).

---
### 🎓 Fin del módulo: de un CSV a un modelo que explica y predice.